# 02 - Clasificación contextual de transacciones

## 1. Dependencias

In [ ]:
#Instalar dependencias
%pip install pandas numpy scikit-learn matplotlib joblib

## 2. Configuración

In [ ]:
#Librerías
import platform
import sys
import unicodedata
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline

In [ ]:
#Configuración
SEMILLA = 20260724
UMBRAL_TRANSACCION = 0.50
UMBRAL_CONFIRMACION = 0.50
VERSION_MODELO = 'fincoach_transacciones_mvp_v2'

versiones = pd.DataFrame([
    {'componente': 'python', 'version': sys.version.split()[0]},
    {'componente': 'plataforma', 'version': platform.platform()},
    {'componente': 'pandas', 'version': pd.__version__},
    {'componente': 'numpy', 'version': np.__version__},
    {'componente': 'scikit-learn', 'version': sklearn.__version__},
])

display(versiones)

## 3. Carga del dataset

In [ ]:
#Rutas
ruta_perfiles = Path('../Datasets/perfil_integral_usuario.csv')
ruta_transacciones = Path('../Datasets/transacciones_contextualizadas.csv')
ruta_modelo_usuario = Path('../Modelos/01_conocimiento_usuario.joblib')

assert ruta_perfiles.exists(), f'No existe {ruta_perfiles}'
assert ruta_transacciones.exists(), f'No existe {ruta_transacciones}'
assert ruta_modelo_usuario.exists(), 'Ejecuta primero el notebook 01 para generar su modelo'

print(f'Perfiles: {ruta_perfiles}')
print(f'Transacciones: {ruta_transacciones}')
print(f'Modelo de usuario: {ruta_modelo_usuario}')

In [ ]:
#Cargar datos
perfiles = pd.read_csv(ruta_perfiles, low_memory=False)
transacciones = pd.read_csv(ruta_transacciones, low_memory=False)
artefacto_usuario = joblib.load(ruta_modelo_usuario)

display(perfiles.head(5))
display(transacciones.head(5))

## 4. Contrato y unión contextual

In [ ]:
#Validar contrato
columnas_transacciones = [
    'transaccion_id',
    'perfil_id',
    'descripcion_original',
    'descripcion_normalizada',
    'valor',
    'direccion',
    'tipo_movimiento',
    'categoria_principal',
    'finalidad',
    'nota_usuario',
    'estado_clasificacion',
    'requiere_confirmacion',
    'regularidad_movimiento',
    'particion',
]

columnas_perfiles = [
    'perfil_id',
    'actividad_principal_declarada',
    'actividad_1_familia',
    'estado_ingreso_actual',
    'ingreso_mensual_neto',
    'ingreso_adicional',
    'objetivo_proximo',
    'hobbies',
    'responsabilidad_financiera',
    'tipos_deuda',
    'nivel_endeudamiento_pct',
    'habito_ahorro',
]

faltantes_transacciones = [
    columna for columna in columnas_transacciones if columna not in transacciones.columns
]

faltantes_perfiles = [
    columna for columna in columnas_perfiles if columna not in perfiles.columns
]

assert not faltantes_transacciones, f'Faltan columnas en transacciones: {faltantes_transacciones}'
assert not faltantes_perfiles, f'Faltan columnas en perfiles: {faltantes_perfiles}'
assert transacciones['transaccion_id'].is_unique, 'transaccion_id contiene duplicados'
assert perfiles['perfil_id'].is_unique, 'perfil_id contiene duplicados'
assert set(transacciones['particion'].unique()) == {'train', 'validation', 'test'}, 'Particiones inválidas'

print('Contrato: OK')

In [ ]:
#Contexto
datos = transacciones.merge(
    perfiles[columnas_perfiles],
    on='perfil_id',
    how='left',
    validate='many_to_one',
)

perfiles_sin_contexto = datos['actividad_principal_declarada'].isna().sum()
cruce_particiones = datos.groupby('perfil_id')['particion'].nunique()

assert perfiles_sin_contexto == 0, 'Existen transacciones sin perfil asociado'
assert cruce_particiones.eq(1).all(), 'Un perfil aparece en más de una partición'

display(pd.DataFrame([{
    'transacciones': len(datos),
    'perfiles': datos['perfil_id'].nunique(),
    'sin_contexto': int(perfiles_sin_contexto),
}]))

## 5. Cobertura de etiquetas

In [ ]:
#Mostrar cobertura
resumen_etiquetas = pd.DataFrame([
    {
        'objetivo': 'categoria_principal',
        'clases': datos['categoria_principal'].nunique(),
        'clase_menor': datos['categoria_principal'].value_counts().idxmin(),
        'filas_clase_menor': int(datos['categoria_principal'].value_counts().min()),
    },
    
    {
        'objetivo': 'finalidad',
        'clases': datos['finalidad'].nunique(),
        'clase_menor': datos['finalidad'].value_counts().idxmin(),
        'filas_clase_menor': int(datos['finalidad'].value_counts().min()),
    },
    
    {
        'objetivo': 'requiere_confirmacion',
        'clases': datos['requiere_confirmacion'].nunique(),
        'clase_menor': datos['requiere_confirmacion'].value_counts().idxmin(),
        'filas_clase_menor': int(datos['requiere_confirmacion'].value_counts().min()),
    },
])

display(resumen_etiquetas)
display(pd.crosstab(datos['categoria_principal'], datos['particion']))

In [ ]:
#Validar cobertura
cobertura_categoria = pd.crosstab(
    datos['categoria_principal'],
    datos['particion'],
).reindex(columns=['train', 'validation', 'test'], fill_value=0)

cobertura_finalidad = pd.crosstab(
    datos['finalidad'],
    datos['particion'],
).reindex(columns=['train', 'validation', 'test'], fill_value=0)

assert cobertura_categoria.gt(0).all(axis=1).all(), 'Alguna categoría no aparece en las tres particiones'
assert cobertura_finalidad.gt(0).all(axis=1).all(), 'Alguna finalidad no aparece en las tres particiones'
print('Cobertura de etiquetas: OK')

## 6. Ingeniería de atributos

In [ ]:
#Preparar contexto
def valor_texto(valor, reemplazo='no_declarado'):
    if pd.isna(valor) or not str(valor).strip():
        return reemplazo
    return str(valor).strip()


datos['actividad_secundaria_modelo'] = datos['ingreso_adicional'].fillna('no_declarada')
datos['hobbies_modelo'] = datos['hobbies'].fillna('no_declarado')
datos['meta_modelo'] = datos['objetivo_proximo'].fillna('no_declarada')
datos['responsabilidad_modelo'] = datos['responsabilidad_financiera'].fillna('no_declarada')
datos['deuda_modelo'] = datos['tipos_deuda'].fillna('sin_deuda')
datos['endeudamiento_modelo'] = datos['nivel_endeudamiento_pct'].fillna('no_calculable')

In [ ]:
#Construir atributos
def construir_texto_modelo(fila):
    partes = [
        'transaccion {}'.format(valor_texto(fila['descripcion_normalizada'])),
        'nota {}'.format(valor_texto(fila['nota_usuario'])),
        'direccion {}'.format(valor_texto(fila['direccion'])),
        'tipo {}'.format(valor_texto(fila['tipo_movimiento'])),
        'actividad {}'.format(valor_texto(fila['actividad_1_familia'])),
        'actividad secundaria {}'.format(valor_texto(fila['actividad_secundaria_modelo'])),
        'estado ingreso {}'.format(valor_texto(fila['estado_ingreso_actual'])),
        'hobbies {}'.format(valor_texto(fila['hobbies_modelo'])),
        'meta {}'.format(valor_texto(fila['meta_modelo'])),
        'responsabilidad {}'.format(valor_texto(fila['responsabilidad_modelo'])),
        'deuda {}'.format(valor_texto(fila['deuda_modelo'])),
        'habito ahorro {}'.format(valor_texto(fila['habito_ahorro'])),
    ]
    return ' | '.join(partes)


mascara_train = datos['particion'].eq('train')
mascara_validation = datos['particion'].eq('validation')
mascara_test = datos['particion'].eq('test')
escala_valor = float(np.log1p(pd.to_numeric(datos.loc[mascara_train, 'valor'])).max())
ingresos_modelo = pd.to_numeric(datos['ingreso_mensual_neto'], errors='coerce').fillna(0).clip(lower=0)
proporcion_ingreso = pd.to_numeric(datos['valor']) / ingresos_modelo.clip(lower=1)
escala_proporcion = float(np.log1p(proporcion_ingreso.loc[mascara_train]).max())

datos['texto_modelo'] = datos.apply(construir_texto_modelo, axis=1)
datos['valor_modelo'] = np.log1p(pd.to_numeric(datos['valor'])) / escala_valor
datos['proporcion_ingreso_modelo'] = np.log1p(proporcion_ingreso) / escala_proporcion

columnas_modelo = ['texto_modelo', 'valor_modelo', 'proporcion_ingreso_modelo']
X_train = datos.loc[mascara_train, columnas_modelo]
X_validation = datos.loc[mascara_validation, columnas_modelo]
X_test = datos.loc[mascara_test, columnas_modelo]

display(datos[columnas_modelo].head(5))

## 7. Categoría principal

In [ ]:
#Definir modelos
def crear_preprocesador():
    return ColumnTransformer([
        (
            'texto',
            TfidfVectorizer(
                analyzer='char_wb',
                ngram_range=(3, 5),
                lowercase=True,
                strip_accents='unicode',
                min_df=2,
                max_features=40000,
                sublinear_tf=True,
            ),
            'texto_modelo',
        ),
        ('numericas', 'passthrough', ['valor_modelo', 'proporcion_ingreso_modelo']),
    ])


def crear_pipeline(clasificador):
    return Pipeline([
        ('preprocesamiento', crear_preprocesador()),
        ('clasificador', clasificador),
    ])


candidatos_categoria = {
    'regresion_logistica_sgd': crear_pipeline(SGDClassifier(
        loss='log_loss',
        alpha=0.00001,
        max_iter=2000,
        class_weight='balanced',
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=10,
        average=True,
        random_state=SEMILLA,
    )),
    'naive_bayes_complementario': crear_pipeline(ComplementNB(alpha=0.35)),
}

In [ ]:
#Comparar categorías
y_categoria_train = datos.loc[mascara_train, 'categoria_principal']
y_categoria_validation = datos.loc[mascara_validation, 'categoria_principal']
y_categoria_test = datos.loc[mascara_test, 'categoria_principal']
metricas_candidatos = []

for nombre, modelo in candidatos_categoria.items():
    modelo.fit(X_train, y_categoria_train)
    prediccion = modelo.predict(X_validation)
    metricas_candidatos.append({
        'modelo': nombre,
        'f1_macro_validation': f1_score(
            y_categoria_validation,
            prediccion,
            average='macro',
            zero_division=0,
        ),
        'accuracy_validation': accuracy_score(y_categoria_validation, prediccion),
    })

metricas_candidatos = pd.DataFrame(metricas_candidatos)
puntajes_categoria = metricas_candidatos.set_index('modelo')['f1_macro_validation'].to_dict()
nombre_modelo_categoria = max(
    candidatos_categoria,
    key=lambda nombre: (puntajes_categoria[nombre], nombre == 'regresion_logistica_sgd'),
)

modelo_categoria = candidatos_categoria[nombre_modelo_categoria]
display(metricas_candidatos.sort_values('f1_macro_validation', ascending=False))
print(f'Modelo seleccionado: {nombre_modelo_categoria}')

In [ ]:
#Evaluar categorías
prediccion_categoria_test = modelo_categoria.predict(X_test)
reporte_categoria = classification_report(
    y_categoria_test,
    prediccion_categoria_test,
    output_dict=True,
    zero_division=0,
)

resumen_categoria = pd.DataFrame([{
    'modelo': nombre_modelo_categoria,
    'f1_macro_test': f1_score(
        y_categoria_test,
        prediccion_categoria_test,
        average='macro',
        zero_division=0,
    ),
    'accuracy_test': accuracy_score(y_categoria_test, prediccion_categoria_test),
    'transacciones_test': len(y_categoria_test),
}])

metricas_categoria = pd.DataFrame(reporte_categoria).T.loc[
    sorted(y_categoria_test.unique()),
    ['precision', 'recall', 'f1-score', 'support'],
]

display(resumen_categoria)
display(metricas_categoria)

In [ ]:
#Mostrar confusión
figura, eje = plt.subplots(figsize=(14, 12))

ConfusionMatrixDisplay.from_predictions(
    y_categoria_test,
    prediccion_categoria_test,
    labels=sorted(y_categoria_test.unique()),
    normalize='true',
    xticks_rotation=90,
    cmap='Blues',
    ax=eje,
    colorbar=False,
)

eje.set_title('Matriz de confusión normalizada - categoría principal')
plt.tight_layout()
plt.show()

## 8. Finalidad y confirmación

In [ ]:
#Entrenar finalidad
modelo_finalidad = crear_pipeline(SGDClassifier(
    loss='log_loss',
    alpha=0.00001,
    max_iter=2000,
    class_weight='balanced',
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=10,
    average=True,
    random_state=SEMILLA,
))

y_finalidad_train = datos.loc[mascara_train, 'finalidad']
y_finalidad_validation = datos.loc[mascara_validation, 'finalidad']
y_finalidad_test = datos.loc[mascara_test, 'finalidad']

modelo_finalidad.fit(X_train, y_finalidad_train)
prediccion_finalidad_validation = modelo_finalidad.predict(X_validation)
prediccion_finalidad_test = modelo_finalidad.predict(X_test)

metricas_finalidad = pd.DataFrame([{
    'f1_macro_validation': f1_score(
        y_finalidad_validation,
        prediccion_finalidad_validation,
        average='macro',
        zero_division=0,
    ),
    'f1_macro_test': f1_score(
        y_finalidad_test,
        prediccion_finalidad_test,
        average='macro',
        zero_division=0,
    ),
    'accuracy_test': accuracy_score(y_finalidad_test, prediccion_finalidad_test),
}])

modelo_regularidad = crear_pipeline(SGDClassifier(
    loss='log_loss',
    alpha=0.00001,
    max_iter=2000,
    class_weight='balanced',
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=10,
    average=True,
    random_state=SEMILLA,
))

y_regularidad_train = datos.loc[mascara_train, 'regularidad_movimiento']
y_regularidad_test = datos.loc[mascara_test, 'regularidad_movimiento']
modelo_regularidad.fit(X_train, y_regularidad_train)
prediccion_regularidad_test = modelo_regularidad.predict(X_test)

metricas_regularidad = pd.DataFrame([{
    'f1_macro_test': f1_score(y_regularidad_test, prediccion_regularidad_test, average='macro', zero_division=0),
    'accuracy_test': accuracy_score(y_regularidad_test, prediccion_regularidad_test),
}])

display(metricas_finalidad)
display(metricas_regularidad)

In [ ]:
#Validar compatibilidad
tabla_compatibilidad = pd.crosstab(
    datos.loc[mascara_train, 'finalidad'],
    datos.loc[mascara_train, 'categoria_principal'],
).gt(0)

categorias_por_finalidad = {
    finalidad: tabla_compatibilidad.columns[fila.to_numpy()].tolist()
    for finalidad, fila in tabla_compatibilidad.iterrows()
}

parejas_validas = set(zip(
    datos.loc[mascara_train, 'categoria_principal'],
    datos.loc[mascara_train, 'finalidad'],
))

compatibilidad_test = [
    (categoria, finalidad) in parejas_validas
    for categoria, finalidad in zip(prediccion_categoria_test, prediccion_finalidad_test)
]

metricas_compatibilidad = pd.DataFrame([{
    'parejas_validas_test_pct': round(float(np.mean(compatibilidad_test)) * 100, 2),
    'parejas_incompatibles_test': int((~np.asarray(compatibilidad_test)).sum()),
}])

display(tabla_compatibilidad)
display(metricas_compatibilidad)

In [ ]:
#Entrenar confirmación
modelo_confirmacion = crear_pipeline(SGDClassifier(
    loss='log_loss',
    alpha=0.00001,
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=10,
    average=True,
    random_state=SEMILLA,
))

y_confirmacion_train = datos.loc[mascara_train, 'requiere_confirmacion']
y_confirmacion_validation = datos.loc[mascara_validation, 'requiere_confirmacion']
y_confirmacion_test = datos.loc[mascara_test, 'requiere_confirmacion']

modelo_confirmacion.fit(X_train, y_confirmacion_train)
probabilidades_confirmacion_validation = modelo_confirmacion.predict_proba(X_validation)
posicion_confirmacion_si = list(modelo_confirmacion.classes_).index('si')
probabilidad_si_validation = probabilidades_confirmacion_validation[:, posicion_confirmacion_si]
resultados_umbrales_confirmacion = []

for umbral in np.arange(0.10, 1.00, 0.01):
    prediccion = np.where(probabilidad_si_validation >= umbral, 'si', 'no')
    resultados_umbrales_confirmacion.append({
        'umbral': round(float(umbral), 2),
        'f1_confirmacion': f1_score(
            y_confirmacion_validation,
            prediccion,
            pos_label='si',
            zero_division=0,
        ),
        'precision_confirmacion': precision_score(
            y_confirmacion_validation,
            prediccion,
            pos_label='si',
            zero_division=0,
        ),
        'recall_confirmacion': recall_score(
            y_confirmacion_validation,
            prediccion,
            pos_label='si',
            zero_division=0,
        ),
    })

resultados_umbrales_confirmacion = pd.DataFrame(resultados_umbrales_confirmacion)
UMBRAL_CONFIRMACION = float(
    resultados_umbrales_confirmacion.sort_values(
        ['f1_confirmacion', 'precision_confirmacion', 'umbral'],
        ascending=[False, False, False],
    ).iloc[0]['umbral']
)

probabilidades_confirmacion_test = modelo_confirmacion.predict_proba(X_test)
probabilidad_si_test = probabilidades_confirmacion_test[:, posicion_confirmacion_si]
prediccion_confirmacion_validation = np.where(
    probabilidad_si_validation >= UMBRAL_CONFIRMACION,
    'si',
    'no',
)
prediccion_confirmacion_test = np.where(
    probabilidad_si_test >= UMBRAL_CONFIRMACION,
    'si',
    'no',
)

reporte_confirmacion = classification_report(
    y_confirmacion_test,
    prediccion_confirmacion_test,
    output_dict=True,
    zero_division=0,
)

metricas_confirmacion = pd.DataFrame([{
    'f1_macro_validation': f1_score(
        y_confirmacion_validation,
        prediccion_confirmacion_validation,
        average='macro',
        zero_division=0,
    ),
    'f1_macro_test': f1_score(
        y_confirmacion_test,
        prediccion_confirmacion_test,
        average='macro',
        zero_division=0,
    ),
    'recall_requiere_confirmacion_test': reporte_confirmacion['si']['recall'],
    'precision_requiere_confirmacion_test': reporte_confirmacion['si']['precision'],
}])

display(resultados_umbrales_confirmacion)
display(metricas_confirmacion)
print(f'Umbral de confirmación seleccionado: {UMBRAL_CONFIRMACION:.2f}')

In [ ]:
#Evaluar abstención
probabilidades_categoria_validation = modelo_categoria.predict_proba(X_validation)
confianza_categoria_validation = probabilidades_categoria_validation.max(axis=1)
aceptadas = confianza_categoria_validation >= UMBRAL_TRANSACCION

exactitud_aceptadas = accuracy_score(
    y_categoria_validation[aceptadas],
    modelo_categoria.predict(X_validation)[aceptadas],
) if aceptadas.any() else np.nan

resumen_abstencion = pd.DataFrame([{
    'umbral': UMBRAL_TRANSACCION,
    'cobertura_validation_pct': round(float(aceptadas.mean()) * 100, 2),
    'abstencion_validation_pct': round(float((~aceptadas).mean()) * 100, 2),
    'accuracy_en_aceptadas': exactitud_aceptadas,
}])

display(resumen_abstencion)

## 9. Uso del modelo de usuario

In [ ]:
#Preparar modelo de usuario
def normalizar_texto(texto):
    texto = unicodedata.normalize('NFKD', str(texto).lower())
    texto = ''.join(caracter for caracter in texto if not unicodedata.combining(caracter))
    texto = texto.replace('_', ' ').replace('-', ' ')
    return ' '.join(texto.split())


def puntuar_actividad_usuario(texto):
    modelo = artefacto_usuario['modelo_actividad']
    probabilidades = dict(zip(modelo.classes_, modelo.predict_proba([texto])[0]))
    vector = artefacto_usuario['vectorizador_catalogo'].transform([texto])
    similitudes = cosine_similarity(vector, artefacto_usuario['matriz_catalogo'])[0]
    familias = artefacto_usuario['familias_catalogo']
    resultados = []

    for actividad in modelo.classes_:
        similitud = float(similitudes[familias == actividad].max())
        probabilidad = float(probabilidades[actividad])
        confianza = (
            artefacto_usuario['peso_modelo'] * probabilidad
            + artefacto_usuario['peso_catalogo'] * similitud
        )
        resultados.append({
            'actividad': str(actividad),
            'confianza': confianza,
        })

    return sorted(resultados, key=lambda resultado: resultado['confianza'], reverse=True)


def clasificar_hobbies_usuario(hobbies):
    if isinstance(hobbies, (list, tuple, set)):
        texto = ' | '.join(str(hobby) for hobby in hobbies)
    else:
        texto = str(hobbies or '')

    texto_normalizado = normalizar_texto(texto)
    palabras = set(texto_normalizado.replace('|', ' ').replace(',', ' ').replace(';', ' ').split())
    encontrados = []

    for hobby, variantes in artefacto_usuario['variantes_hobbies'].items():
        for variante in variantes:
            variante_normalizada = normalizar_texto(variante)
            coincide = (
                variante_normalizada in texto_normalizado
                if ' ' in variante_normalizada
                else variante_normalizada in palabras
            )
            if coincide:
                encontrados.append(hobby)
                break

    return encontrados if encontrados else ['no_declarado']

In [ ]:
#Clasificar usuario
def conocer_usuario(test_usuario):
    if not isinstance(test_usuario, dict):
        raise TypeError('test_usuario debe ser un diccionario')

    actividad_declarada = str(
        test_usuario.get('actividad_principal_declarada')
        or test_usuario.get('ingreso_principal')
        or ''
    ).strip()
    if not actividad_declarada:
        raise ValueError('actividad_principal_declarada debe contener texto')

    resultados = puntuar_actividad_usuario(actividad_declarada)
    mejor_resultado = resultados[0]
    confianza = mejor_resultado['confianza']
    dentro_mvp = confianza >= artefacto_usuario['umbral_alcance']
    actividad = mejor_resultado['actividad'] if dentro_mvp else 'no_disponible'

    ingreso_mensual = float(test_usuario.get('ingreso_mensual_neto', 0))
    modalidad = str(test_usuario.get('modalidad_ingreso_principal', '')).strip().lower()
    estado_ingreso = 'sin_ingresos' if ingreso_mensual == 0 else modalidad
    if estado_ingreso not in artefacto_usuario['modalidades_ingreso']:
        raise ValueError('modalidad_ingreso_principal no es válida')

    ingreso_adicional = str(test_usuario.get('ingreso_adicional', '') or '').strip()
    actividad_secundaria = 'no_declarada'
    if test_usuario.get('tiene_ingreso_adicional') and ingreso_adicional:
        resultado_secundario = puntuar_actividad_usuario(ingreso_adicional)[0]
        actividad_secundaria = (
            resultado_secundario['actividad']
            if resultado_secundario['confianza'] >= artefacto_usuario['umbral_alcance']
            else 'fuera_del_mvp'
        )

    tipos_deuda = test_usuario.get('tipos_deuda', [])
    if isinstance(tipos_deuda, (list, tuple, set)):
        tipos_deuda = ' | '.join(str(deuda) for deuda in tipos_deuda)

    return {
        'estado_alcance_mvp': 'dentro_del_mvp' if dentro_mvp else 'no_disponible',
        'actividad_principal': actividad,
        'actividad_secundaria': actividad_secundaria,
        'estado_ingreso_actual': estado_ingreso,
        'ingreso_mensual_neto': ingreso_mensual,
        'hobbies_intereses': clasificar_hobbies_usuario(test_usuario.get('hobbies', [])),
        'meta': str(test_usuario.get('objetivo_proximo', '') or 'no_declarada').strip(),
        'responsabilidad': str(test_usuario.get('responsabilidad_financiera', '') or 'no_declarada').strip(),
        'tipos_deuda': str(tipos_deuda or 'sin_deuda'),
        'nivel_endeudamiento_pct': test_usuario.get('nivel_endeudamiento_pct'),
        'habito_ahorro': str(test_usuario.get('habito_ahorro', 'no_declarado')).strip().lower(),
        'confianza_usuario_pct': round(confianza * 100, 2),
    }

## 10. Clasificación completa

In [ ]:
#Clasificar transacción
def detectar_tipo_movimiento(descripcion, direccion, tipo_declarado=''):
    palabras = set(normalizar_texto(descripcion).split())
    es_pago = bool({'pago', 'cuota'} & palabras)
    es_deuda = bool({'credito', 'prestamo'} & palabras)

    if direccion == 'salida' and es_pago and es_deuda:
        return 'pago_deuda'
    if tipo_declarado:
        return tipo_declarado
    return 'ingreso_generado' if direccion == 'entrada' else 'gasto'


def clasificar_transaccion(test_usuario, test_transaccion):
    if not isinstance(test_transaccion, dict):
        raise TypeError('test_transaccion debe ser un diccionario')

    contexto = conocer_usuario(test_usuario)
    descripcion = str(test_transaccion.get('descripcion', '')).strip()
    nota_usuario = str(test_transaccion.get('nota_usuario', '') or '').strip()
    valor_numerico = float(test_transaccion.get('valor', 0))
    direccion = str(test_transaccion.get('direccion', 'salida')).strip().lower()

    if not descripcion:
        raise ValueError('descripcion debe contener texto')
    if valor_numerico <= 0:
        raise ValueError('valor debe ser mayor que cero')
    if direccion not in ['entrada', 'salida']:
        raise ValueError('direccion debe ser entrada o salida')

    tipo_movimiento = detectar_tipo_movimiento(
        descripcion,
        direccion,
        str(test_transaccion.get('tipo_movimiento', '') or '').strip(),
    )
    hobbies_modelo = ' | '.join(contexto['hobbies_intereses'])
    fila = {
        'descripcion_normalizada': normalizar_texto(descripcion),
        'nota_usuario': nota_usuario,
        'direccion': direccion,
        'tipo_movimiento': tipo_movimiento,
        'actividad_1_familia': contexto['actividad_principal'],
        'actividad_secundaria_modelo': contexto['actividad_secundaria'],
        'estado_ingreso_actual': contexto['estado_ingreso_actual'],
        'hobbies_modelo': hobbies_modelo,
        'meta_modelo': contexto['meta'],
        'responsabilidad_modelo': contexto['responsabilidad'],
        'deuda_modelo': contexto['tipos_deuda'],
        'habito_ahorro': contexto['habito_ahorro'],
    }
    proporcion_ingreso = valor_numerico / max(contexto['ingreso_mensual_neto'], 1)
    entrada = pd.DataFrame([{
        'texto_modelo': construir_texto_modelo(fila),
        'valor_modelo': np.log1p(valor_numerico) / escala_valor,
        'proporcion_ingreso_modelo': np.log1p(proporcion_ingreso) / escala_proporcion,
    }])

    probabilidades_categoria = modelo_categoria.predict_proba(entrada)[0]
    probabilidades_finalidad = modelo_finalidad.predict_proba(entrada)[0]
    probabilidades_regularidad = modelo_regularidad.predict_proba(entrada)[0]
    regla_aplicada = 'modelo_contextual'

    if tipo_movimiento == 'pago_deuda':
        probabilidades_categoria = np.zeros_like(probabilidades_categoria)
        posicion_deuda = list(modelo_categoria.classes_).index('Deuda y financiación')
        probabilidades_categoria[posicion_deuda] = 1.0
        finalidad = 'pago_deuda'
        confianza_finalidad = 1.0
        regla_aplicada = 'pago_deuda_explicito'
    else:
        posicion_finalidad = int(np.argmax(probabilidades_finalidad))
        finalidad = str(modelo_finalidad.classes_[posicion_finalidad])
        confianza_finalidad = float(probabilidades_finalidad[posicion_finalidad])

    orden = np.argsort(probabilidades_categoria)[::-1]
    categoria_principal = str(modelo_categoria.classes_[orden[0]])
    confianza_categoria = float(probabilidades_categoria[orden[0]])
    top_3_categorias = [
        {
            'categoria': str(modelo_categoria.classes_[posicion]),
            'porcentaje': round(float(probabilidades_categoria[posicion]) * 100, 2),
        }
        for posicion in orden[:3]
        if probabilidades_categoria[posicion] > 0
    ]
    porcentajes_categorias = {
        str(modelo_categoria.classes_[posicion]): round(float(probabilidades_categoria[posicion]) * 100, 2)
        for posicion in orden
    }

    posicion_regularidad = int(np.argmax(probabilidades_regularidad))
    regularidad = str(modelo_regularidad.classes_[posicion_regularidad])
    confianza_regularidad = float(probabilidades_regularidad[posicion_regularidad])

    probabilidades_confirmacion = modelo_confirmacion.predict_proba(entrada)[0]
    posicion_si = list(modelo_confirmacion.classes_).index('si')
    probabilidad_confirmacion = float(probabilidades_confirmacion[posicion_si])
    pareja_compatible = (categoria_principal, finalidad) in parejas_validas
    categoria_contextual = categoria_principal in ['Ocio', 'Inversión productiva', 'Trabajo independiente']

    requiere_confirmacion = (
        confianza_categoria < UMBRAL_TRANSACCION
        or categoria_principal == 'Otra / ambigua'
        or probabilidad_confirmacion >= UMBRAL_CONFIRMACION
        or not pareja_compatible
        or (contexto['estado_alcance_mvp'] == 'no_disponible' and categoria_contextual)
    )
    if regla_aplicada == 'pago_deuda_explicito':
        requiere_confirmacion = False
        probabilidad_confirmacion = 0.0

    return {
        'descripcion_transaccion': descripcion,
        'nota_usuario': nota_usuario,
        'valor': valor_numerico,
        'direccion': direccion,
        'tipo_movimiento': tipo_movimiento,
        'estado_alcance_usuario': contexto['estado_alcance_mvp'],
        'actividad_usuario': contexto['actividad_principal'],
        'estado_ingreso_usuario': contexto['estado_ingreso_actual'],
        'hobbies_usuario': contexto['hobbies_intereses'],
        'categoria_principal': categoria_principal,
        'confianza_categoria_pct': round(confianza_categoria * 100, 2),
        'porcentajes_categorias': porcentajes_categorias,
        'top_3_categorias': top_3_categorias,
        'finalidad': finalidad,
        'confianza_finalidad_pct': round(confianza_finalidad * 100, 2),
        'regularidad_movimiento': regularidad,
        'confianza_regularidad_pct': round(confianza_regularidad * 100, 2),
        'requiere_confirmacion': 'si' if requiere_confirmacion else 'no',
        'probabilidad_confirmacion_pct': round(probabilidad_confirmacion * 100, 2),
        'estado_clasificacion': 'requiere_confirmacion' if requiere_confirmacion else 'clasificada',
        'pareja_categoria_finalidad_valida': pareja_compatible,
        'regla_aplicada': regla_aplicada,
        'advertencia_contexto': (
            'La actividad del usuario está fuera del catálogo profesional del MVP'
            if contexto['estado_alcance_mvp'] == 'no_disponible'
            else 'Contexto del usuario reconocido dentro del MVP'
        ),
        'version_modelo': VERSION_MODELO,
    }

## 11. Serialización

In [ ]:
#Guardar modelo
ruta_modelo_transacciones = Path('../Modelos/02_clasificacion_transacciones.joblib')
ruta_modelo_transacciones.parent.mkdir(parents=True, exist_ok=True)

artefacto_transacciones = {
    'version_modelo': VERSION_MODELO,
    'modelo_categoria': modelo_categoria,
    'nombre_modelo_categoria': nombre_modelo_categoria,
    'modelo_finalidad': modelo_finalidad,
    'modelo_regularidad': modelo_regularidad,
    'modelo_confirmacion': modelo_confirmacion,
    'umbral_transaccion': UMBRAL_TRANSACCION,
    'umbral_confirmacion': UMBRAL_CONFIRMACION,
    'categorias_por_finalidad': categorias_por_finalidad,
    'parejas_validas': parejas_validas,
    'escala_valor': escala_valor,
    'escala_proporcion': escala_proporcion,
    'columnas_modelo': columnas_modelo,
    'version_modelo_usuario': artefacto_usuario['version_modelo'],
}

joblib.dump(artefacto_transacciones, ruta_modelo_transacciones)
print(f'Modelo guardado en: {ruta_modelo_transacciones}')

## 12. Pruebas manuales

In [ ]:
#Probar casos clasificables
usuario_editor = {
    'actividad_principal_declarada': 'Editor de video',
    'ingreso_mensual_neto': 4200000,
    'modalidad_ingreso_principal': 'variable',
    'tiene_ingreso_adicional': False,
    'ingreso_adicional': '',
    'objetivo_proximo': 'comprar equipo de trabajo',
    'hobbies': ['turismo y viajes'],
    'responsabilidad_financiera': 'aporte al hogar',
    'tipos_deuda': ['credito'],
    'nivel_endeudamiento_pct': 18,
    'habito_ahorro': 'media',
}

usuario_software = {
    'actividad_principal_declarada': 'Desarrollador de software',
    'ingreso_mensual_neto': 5500000,
    'modalidad_ingreso_principal': 'fijo',
    'tiene_ingreso_adicional': False,
    'ingreso_adicional': '',
    'objetivo_proximo': 'crear un fondo de emergencia',
    'hobbies': ['videojuegos y streaming'],
    'responsabilidad_financiera': '',
    'tipos_deuda': [],
    'nivel_endeudamiento_pct': 0,
    'habito_ahorro': 'alta',
}

usuario_nutricion = {
    'actividad_principal_declarada': 'Nutricionista',
    'ingreso_mensual_neto': 2800000,
    'modalidad_ingreso_principal': 'mixto',
    'tiene_ingreso_adicional': False,
    'ingreso_adicional': '',
    'objetivo_proximo': 'ampliar actividad economica',
    'hobbies': ['cocina y reposteria'],
    'responsabilidad_financiera': 'gastos de un hijo',
    'tipos_deuda': ['credito'],
    'nivel_endeudamiento_pct': 22,
    'habito_ahorro': 'baja',
}

test_transacciones = [
    {
        'test_usuario': usuario_editor,
        'test_transaccion': {'descripcion': 'COMPRA COMPUTADOR', 'valor': 4500000, 'direccion': 'salida', 'nota_usuario': 'Herramienta para mi trabajo de edicion de video'},
        'categoria_esperada': 'Inversión productiva',
    },
    {
        'test_usuario': usuario_software,
        'test_transaccion': {'descripcion': 'COMPRA COMPUTADOR', 'valor': 4500000, 'direccion': 'salida', 'nota_usuario': 'Para videojuegos y streaming en mi tiempo libre'},
        'categoria_esperada': 'Ocio',
    },
    {
        'test_usuario': usuario_nutricion,
        'test_transaccion': {'descripcion': 'COMPRA MERCADO', 'valor': 180000, 'direccion': 'salida', 'nota_usuario': ''},
        'categoria_esperada': 'Alimentación',
    },
    {
        'test_usuario': usuario_nutricion,
        'test_transaccion': {'descripcion': 'PAGO CREDITO', 'valor': 150000, 'direccion': 'salida', 'nota_usuario': 'credito educativo'},
        'categoria_esperada': 'Deuda y financiación',
    },
]

resultados_transacciones = []

for caso in test_transacciones:
    resultado = clasificar_transaccion(caso['test_usuario'], caso['test_transaccion'])
    resultado['categoria_esperada'] = caso['categoria_esperada']
    resultado['cumple_esperado'] = (
        resultado['categoria_principal'] == caso['categoria_esperada']
        and resultado['estado_clasificacion'] == 'clasificada'
    )
    resultados_transacciones.append(resultado)

columnas_prueba = [
    'descripcion_transaccion',
    'actividad_usuario',
    'hobbies_usuario',
    'categoria_principal',
    'categoria_esperada',
    'confianza_categoria_pct',
    'top_3_categorias',
    'porcentajes_categorias',
    'finalidad',
    'confianza_finalidad_pct',
    'regularidad_movimiento',
    'requiere_confirmacion',
    'cumple_esperado',
]

resultados_transacciones = pd.DataFrame(resultados_transacciones)
display(resultados_transacciones[columnas_prueba])
assert resultados_transacciones['cumple_esperado'].all(), 'Uno de los casos clasificables no cumplió la salida esperada'

In [ ]:
#Probar caso ambiguo
test_ambiguo = {
    'descripcion': 'PAGO QR',
    'valor': 85000,
    'direccion': 'salida',
    'nota_usuario': '',
    'salida_esperada': 'requiere_confirmacion',
}

salida_esperada = test_ambiguo['salida_esperada']
test_transaccion = {clave: valor for clave, valor in test_ambiguo.items() if clave != 'salida_esperada'}
resultado_ambiguo = clasificar_transaccion(usuario_software, test_transaccion)
resultado_ambiguo['salida_esperada'] = salida_esperada
resultado_ambiguo['cumple_esperado'] = (
    resultado_ambiguo['estado_clasificacion'] == salida_esperada
)

display(pd.DataFrame([resultado_ambiguo]))
assert resultado_ambiguo['cumple_esperado'], 'El caso ambiguo no solicitó confirmación'